In [ ]:
import pandas as pd
from pipeline import IndicatorSpec, build_feature_df
from data import fetch_binance_klines
from simulation.rules import Rule, RuleGroup, ALL, ANY
from simulation.rule_features import add_last_n_compare, add_cross_compare
from simulation.simulator import Strategy, SimConfig, run_simulation, align_timeframes
from plot_simulation import plot_simulation
from plotter import plot_interactive, PlotConfig, IndicatorSpec
from mtf_plot import make_plot_indicators, assert_plot_columns_exist
from features.ema_spreads import EmaSpreadSpec, add_ema_spread_features
from features.cross_through import CrossThroughSpec, add_cross_through_features
from simulation.rules import Ctx
from analytics import (
    trades_to_frame,
    plot_balance_by_trade,
    plot_trade_pnl_bars,
    trade_summary_table
)
from plot_toggles import (
    PlotToggle,
    make_plot_indicators_from_toggles,
    assert_plot_columns_exist,
)

from ema_diagnostic_plots import (
    ema_pair_spread_panel,
    ema_group_spread_panel,
    ema_cut_through_panel,
)

In [ ]:
symbol = "BTCUSDT"
market = "spot"
tz = "Asia/Karachi"

# Start earlier for EMA warmup.
fetch_start = "2026-01-01"
sim_start = "2026-05-06"
end = None

In [ ]:
# ============================================================
# RSI Configs
# ============================================================

RSI_DIV_CFG = {
    "length": 14,
    "pivot_lookback": 5,
    "min_rsi_delta": 2,

    # divergence filtering
    "os_level": 30,
    "ob_level": 70,
    "zone_mode": "cross",

    # panel styling
    "major_levels": [30, 70],
    "minor_levels": [20, 80],
    "show_zone_shading": True,

    # divergence labels
    "show_div_labels": True,
    "max_div_labels": 6,
    "label_font_size": 10,
    "label_xshift": 10,
    "label_yshift": 14,

    # main chart markers
    "mark_price": True,
    "price_marker_size": 18,
    "marker_offset_mult": 1.35,
}

# ============================================================
# Supertrend Configs
# ============================================================

ST1_1M_CFG = {
    "length": 21,
    "multiplier": 1.0,
    "show_markers": True,
}

ST1_5M_CFG = {
    "length": 21,
    "multiplier": 1.0,
    "show_markers": True,
}

ST1_15M_CFG = {
    "length": 21,
    "multiplier": 1.0,
    "show_markers": True,
}

ST2_1M_CFG = {
    "length": 14,
    "multiplier": 2.0,
    "show_markers": True,
}

ST2_5M_CFG = {
    "length": 14,
    "multiplier": 2.0,
    "show_markers": True,
}

ST2_15M_CFG = {
    "length": 14,
    "multiplier": 2.0,
    "show_markers": True,
}


# ============================================================
# Supertrend Tags
# ============================================================

ST1_TAG = "st1_21_1"
ST2_TAG = "st2_14_2"

In [ ]:
### 1 Minute Indicators ###
EMA50_1m = "1m__ema__EMA_50"
EMA100_1m = "1m__ema__EMA_100"
EMA150_1m = "1m__ema__EMA_150"
EMA200_1m = "1m__ema__EMA_200"
MACD_1M = "1m__macd_8_21_5__MACD"
MACD_SIGNAL_1M = "1m__macd_8_21_5__SIGNAL"
MACD_HIST_1M = "1m__macd_8_21_5__HIST"
BULL_DIV_1m = "1m__rsi14__BULL_DIV"
BEAR_DIV_1m = "1m__rsi14__BEAR_DIV"
BULL_START_RSI_1m = "1m__rsi14__BULL_START_RSI"
BEAR_START_RSI_1m = "1m__rsi14__BEAR_START_RSI"

### 5 Minute Indicators ###
EMA50_5m = "5m__ema__EMA_50"
EMA75_5m = "5m__ema__EMA_75"
EMA100_5m = "5m__ema__EMA_100"
EMA125_5m = "5m__ema__EMA_125"
EMA150_5m = "5m__ema__EMA_150"
EMA175_5m = "5m__ema__EMA_175"
EMA200_5m = "5m__ema__EMA_200"
MACD_5M = "5m__macd_8_21_5__MACD"
MACD_SIGNAL_5M = "5m__macd_8_21_5__SIGNAL"
MACD_HIST_5M = "5m__macd_8_21_5__HIST"
BULL_DIV_5m = "5m__rsi14__BULL_DIV"
BEAR_DIV_5m = "5m__rsi14__BEAR_DIV"
BULL_START_RSI_5m = "5m__rsi14__BULL_START_RSI"
BEAR_START_RSI_5m = "5m__rsi14__BEAR_START_RSI"


### 15 Minute Indicators ###
EMA50_15m = "15m__ema__EMA_50"
EMA75_15m = "15m__ema__EMA_75"
EMA100_15m = "15m__ema__EMA_100"
EMA125_15m = "15m__ema__EMA_125"
EMA150_15m = "15m__ema__EMA_150"
EMA175_15m = "15m__ema__EMA_175"
EMA200_15m = "15m__ema__EMA_200"
MACD_15M = "15m__macd_8_21_5__MACD"
MACD_SIGNAL_15M = "15m__macd_8_21_5__SIGNAL"
MACD_HIST_15M = "15m__macd_8_21_5__HIST"
BULL_DIV_15m = "15m__rsi14__BULL_DIV"
BEAR_DIV_15m = "15m__rsi14__BEAR_DIV"
BULL_START_RSI_15m = "15m__rsi14__BULL_START_RSI"
BEAR_START_RSI_15m = "15m__rsi14__BEAR_START_RSI"



In [ ]:
# ============================================================
# 1m indicators
# ============================================================

ind_1m = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [50, 100, 150, 200, 250],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("rsi_divergence", tag="rsi14", config=RSI_DIV_CFG),

    IndicatorSpec("supertrend", tag=ST1_TAG, config=ST1_1M_CFG),
    IndicatorSpec("supertrend", tag=ST2_TAG, config=ST2_1M_CFG),
]


# ============================================================
# 5m indicators
# ============================================================

ind_5m = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [50, 75, 100, 125, 150, 175, 200],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("rsi_divergence", tag="rsi14", config=RSI_DIV_CFG),

    IndicatorSpec("supertrend", tag=ST1_TAG, config=ST1_5M_CFG),
    IndicatorSpec("supertrend", tag=ST2_TAG, config=ST2_5M_CFG),
]


# ============================================================
# 15m indicators
# ============================================================

ind_15m = [
    IndicatorSpec("moving_average", tag="ema", config={
        "type": "ema",
        "periods": [50, 75, 100, 125, 150, 175, 200],
        "source": "close",
    }),

    IndicatorSpec("macd", tag="macd_8_21_5", config={
        "fast": 8,
        "slow": 21,
        "signal": 5,
    }),

    IndicatorSpec("rsi_divergence", tag="rsi14", config=RSI_DIV_CFG),

    IndicatorSpec("supertrend", tag=ST1_TAG, config=ST1_15M_CFG),
    IndicatorSpec("supertrend", tag=ST2_TAG, config=ST2_15M_CFG),
]

In [ ]:
# 1m base data
df1 = fetch_binance_klines(
    symbol=symbol,
    interval="1m",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df1_feat, _, _ = build_feature_df(
    raw_df=df1,
    tz=tz,
    ma_windows=[],
    indicators=ind_1m,
)


# 5m filter data
df5 = fetch_binance_klines(
    symbol=symbol,
    interval="5m",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df5_feat, _, _ = build_feature_df(
    raw_df=df5,
    tz=tz,
    ma_windows=[],
    indicators=ind_5m,
)


# 15m filter data
df15 = fetch_binance_klines(
    symbol=symbol,
    interval="15m",
    start=fetch_start,
    end=end,
    market=market,
    cache=True,
    cache_dir="data",
)

df15_feat, _, _ = build_feature_df(
    raw_df=df15,
    tz=tz,
    ma_windows=[],
    indicators=ind_15m,
)

merged = align_timeframes(
    base_df=df1_feat,
    other_dfs={
        "5m": df5_feat,
        "15m": df15_feat,
    },
    base_label="1m",
)


In [ ]:
[c for c in merged.columns if "supertrend" in c.lower()]

In [ ]:
N_CONFIRM = 1             # number of consecutive confirmation candles required
MAX_WAIT_AFTER_CROSS = 30   # how many candles after the cross we are willing to wait
MACD_THRESHOLD_1M = 10
MACD_THRESHOLD_5M = 10
MACD_THRESHOLD_15M = 50 

MIN_MACD_DIFF_1M = 5
MIN_MACD_DIFF_5M = 5
MIN_MACD_DIFF_15M = 5

DIV_LOOKBACK = 5

N_SPREAD_CONFIRM = 3
MIN_SPREAD_PCT = 0.1

MIN_EMAS_CROSSED = 4
CROSS_LOOKBACK = 40


#####################################################################################


LONG_EMA_FILTERS = [
    # EMA50_1m,
    # EMA100_1m,
    # EMA150_1m,
    # EMA200_1m,
    # EMA100_5m,
    # EMA200_5m,
    # EMA100_15m,
    EMA200_15m
]

supertrend_long_filter_strict = ALL(

    Rule(
        "Current close above all EMA filters",
        lambda c: c.close_above_all(LONG_EMA_FILTERS)
    ),
    # Rule("1m Supertrend bullish", lambda c: c.gt(ST_DIR_1M, 0)),
    Rule("5m Supertrend bullish", lambda c: c.gt(ST_DIR_5M, 0)),
    #Rule("15m Supertrend bullish", lambda c: c.gt(ST_DIR_15M, 0)),
)

close_rules_long = ALL(
    # Rule(
    #     "1m Supertrend turned bearish",
    #     lambda c: c.lt(ST_DIR_1M, 0)
    # ),
    Rule(
        "1m Supertrend turned bearish",
        lambda c: c.lt(ST_DIR_5M, 0)
    ),
    # Rule(
    #     "1m Supertrend turned bearish",
    #     lambda c: c.lt(ST_DIR_15M, 0)
    # )
)



###################### short rules #########################

# ============================================================
# Helper to build Supertrend column names
# ============================================================

def st_cols(tf: str, tag: str) -> dict:
    root = f"{tf}__{tag}"
    return {
        "ST": f"{root}__ST",
        "DIR": f"{root}__DIR",
        "BUY": f"{root}__BUY",
        "SELL": f"{root}__SELL",
        "BUY_LINE": f"{root}__ST_BUY_LINE",
        "SELL_LINE": f"{root}__ST_SELL_LINE",
    }


# ============================================================
# ST1 aliases: length 21, multiplier 1.0
# ============================================================

ST1_1M = st_cols("1m", ST1_TAG)
ST1_5M = st_cols("5m", ST1_TAG)
ST1_15M = st_cols("15m", ST1_TAG)


# ============================================================
# ST2 aliases: length 14, multiplier 2.0
# ============================================================

ST2_1M = st_cols("1m", ST2_TAG)
ST2_5M = st_cols("5m", ST2_TAG)
ST2_15M = st_cols("15m", ST2_TAG)


In [ ]:
# ============================================================
# STRATEGY COMBINATION OPTIMIZER
# EMA filters + MACD thresholds + Supertrend filters
# ============================================================

import time
import itertools
from pathlib import Path
from dataclasses import asdict, is_dataclass

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from simulation.rules import Rule, ALL
from simulation.simulator import Strategy, SimConfig, run_simulation


# ============================================================
# Config
# ============================================================

INITIAL_CASH = 10_000
RESULTS_PATH = Path("optimization_results_combo_strategy.csv")
CHECKPOINT_EVERY = 25
MIN_TRADES_FOR_RANKING = 20

SIM_START = "2026-01-01"
SIM_END = "2026-05-18"

# Green candle confirmation
N_CONFIRM_VALUES = 2

# MACD thresholds to test
MACD_1M_THRESHOLDS = (0, 5, 10, 20)
MACD_5M_THRESHOLDS = (0, 5, 10, 20)

HIST_1M_THRESHOLDS = (0, 20, 50, 80)
HIST_5M_THRESHOLDS = (0, 20, 50, 80)
HIST_15M_THRESHOLDS = (0, 20, 50, 80)


# ============================================================
# Column aliases
# Make sure these already exist in merged
# ============================================================

# ============================================================
# EMA filters - only 3 options as requested
# ============================================================

EMA_FILTER_LIBRARY = {
    "ema200_1m": [
        EMA200_1m,
    ],

    "ema200_5m": [
        EMA200_5m,
    ],

    "ema200_15m": [
        EMA200_15m,
    ],
}


# ============================================================
# MACD rule library
# ============================================================

MACD_RULE_LIBRARY = {
    "macd1m_hist1m_hist5m": [
        ("1m_macd", MACD_1M, "macd_1m_threshold"),
        ("1m_hist", MACD_HIST_1M, "hist_1m_threshold"),
        ("5m_hist", MACD_HIST_5M, "hist_5m_threshold"),
    ],

    "hist1m_hist5m": [
        ("1m_hist", MACD_HIST_1M, "hist_1m_threshold"),
        ("5m_hist", MACD_HIST_5M, "hist_5m_threshold"),
    ],

    "macd1m_hist1m": [
        ("1m_macd", MACD_1M, "macd_1m_threshold"),
        ("1m_hist", MACD_HIST_1M, "hist_1m_threshold"),
    ],

    "macd1m_macd5m_hist1m_hist5m": [
        ("1m_macd", MACD_1M, "macd_1m_threshold"),
        ("5m_macd", MACD_5M, "macd_5m_threshold"),
        ("1m_hist", MACD_HIST_1M, "hist_1m_threshold"),
        ("5m_hist", MACD_HIST_5M, "hist_5m_threshold"),
    ],

    "hist1m_hist5m_hist15m": [
        ("1m_hist", MACD_HIST_1M, "hist_1m_threshold"),
        ("5m_hist", MACD_HIST_5M, "hist_5m_threshold"),
        ("15m_hist", MACD_HIST_15M, "hist_15m_threshold"),
    ],
}

# ============================================================
# MACD parameter combinations
# Important: this avoids multiplying unused threshold columns
# ============================================================

MACD_1M_THRESHOLDS = (0, 10)
MACD_5M_THRESHOLDS = (0, 10)

HIST_1M_THRESHOLDS = (20, 50, 80)
HIST_5M_THRESHOLDS = (20, 50)
HIST_15M_THRESHOLDS = (20, 50)


MACD_CONFIGS = []


def add_macd_config(rule_key: str, cfg_id: str, **thresholds):
    MACD_CONFIGS.append({
        "macd_rule_key": rule_key,
        "macd_cfg_id": cfg_id,
        "thresholds": thresholds,
    })


# 12 configs
for m1 in MACD_1M_THRESHOLDS:
    for h1 in HIST_1M_THRESHOLDS:
        for h5 in HIST_5M_THRESHOLDS:
            add_macd_config(
                "macd1m_hist1m_hist5m",
                f"m1_{m1}_h1_{h1}_h5_{h5}",
                macd_1m_threshold=m1,
                hist_1m_threshold=h1,
                hist_5m_threshold=h5,
            )


# 6 configs
for h1 in HIST_1M_THRESHOLDS:
    for h5 in HIST_5M_THRESHOLDS:
        add_macd_config(
            "hist1m_hist5m",
            f"h1_{h1}_h5_{h5}",
            hist_1m_threshold=h1,
            hist_5m_threshold=h5,
        )


# 6 configs
for m1 in MACD_1M_THRESHOLDS:
    for h1 in HIST_1M_THRESHOLDS:
        add_macd_config(
            "macd1m_hist1m",
            f"m1_{m1}_h1_{h1}",
            macd_1m_threshold=m1,
            hist_1m_threshold=h1,
        )


# 24 configs
for m1 in MACD_1M_THRESHOLDS:
    for m5 in MACD_5M_THRESHOLDS:
        for h1 in HIST_1M_THRESHOLDS:
            for h5 in HIST_5M_THRESHOLDS:
                add_macd_config(
                    "macd1m_macd5m_hist1m_hist5m",
                    f"m1_{m1}_m5_{m5}_h1_{h1}_h5_{h5}",
                    macd_1m_threshold=m1,
                    macd_5m_threshold=m5,
                    hist_1m_threshold=h1,
                    hist_5m_threshold=h5,
                )


# 12 configs
for h1 in HIST_1M_THRESHOLDS:
    for h5 in HIST_5M_THRESHOLDS:
        for h15 in HIST_15M_THRESHOLDS:
            add_macd_config(
                "hist1m_hist5m_hist15m",
                f"h1_{h1}_h5_{h5}_h15_{h15}",
                hist_1m_threshold=h1,
                hist_5m_threshold=h5,
                hist_15m_threshold=h15,
            )


print(f"MACD configs: {len(MACD_CONFIGS):,}")

# ============================================================
# Supertrend open filters - focused but still broad
# ============================================================

SUPER_OPEN_LIBRARY = {
    "st1st2_1m": [
        ST1_1M["DIR"],
        ST2_1M["DIR"],
    ],

    "st1st2_5m": [
        ST1_5M["DIR"],
        ST2_5M["DIR"],
    ],

    "st1st2_15m": [
        ST1_15M["DIR"],
        ST2_15M["DIR"],
    ],

    "st1st2_1m_5m": [
        ST1_1M["DIR"],
        ST2_1M["DIR"],
        ST1_5M["DIR"],
        ST2_5M["DIR"],
    ],

    "st1st2_1m_15m": [
        ST1_1M["DIR"],
        ST2_1M["DIR"],
        ST1_15M["DIR"],
        ST2_15M["DIR"],
    ],

    "st1st2_1m_5m_15m": [
        ST1_1M["DIR"],
        ST2_1M["DIR"],
        ST1_5M["DIR"],
        ST2_5M["DIR"],
        ST1_15M["DIR"],
        ST2_15M["DIR"],
    ],
}

# ============================================================
# Close rules - same as before
# ============================================================

CLOSE_RULE_LIBRARY = {
    "st2_1m_bear": [
        ST2_1M["DIR"],
    ],

    "st2_5m_bear": [
        ST2_5M["DIR"],
    ],

    "st1_or_st2_1m_bear": [
        ST1_1M["DIR"],
        ST2_1M["DIR"],
    ],

    "st1_or_st2_5m_bear": [
        ST1_5M["DIR"],
        ST2_5M["DIR"],
    ],
}

# ============================================================
# Helpers
# ============================================================

def rolling_all_bool(mask, n: int) -> np.ndarray:
    mask = pd.Series(mask).fillna(False).astype(bool)

    if int(n) <= 1:
        return mask.to_numpy()

    return mask.rolling(int(n), min_periods=int(n)).sum().eq(int(n)).to_numpy()


def all_cols_gt_zero(df: pd.DataFrame, cols) -> np.ndarray:
    return np.logical_and.reduce([
        df[c].to_numpy(dtype=float) > 0
        for c in cols
    ])


def any_cols_lt_zero(df: pd.DataFrame, cols) -> np.ndarray:
    return np.logical_or.reduce([
        df[c].to_numpy(dtype=float) < 0
        for c in cols
    ])


def close_above_all_cols(df: pd.DataFrame, cols) -> np.ndarray:
    close = df["close"].to_numpy(dtype=float)

    return np.logical_and.reduce([
        close > df[c].to_numpy(dtype=float)
        for c in cols
    ])


def build_macd_signal(df: pd.DataFrame, macd_rule_key: str, thresholds: dict) -> np.ndarray:
    parts = []

    for _, col, threshold_key in MACD_RULE_LIBRARY[macd_rule_key]:
        threshold = thresholds[threshold_key]
        parts.append(df[col].to_numpy(dtype=float) > float(threshold))

    return np.logical_and.reduce(parts)


def validate_columns(df: pd.DataFrame) -> None:
    required = set(["open", "close", "t"])

    for cols in EMA_FILTER_LIBRARY.values():
        required.update(cols)

    for rule_items in MACD_RULE_LIBRARY.values():
        for _, col, _ in rule_items:
            required.add(col)

    for cols in SUPER_OPEN_LIBRARY.values():
        required.update(cols)

    for cols in CLOSE_RULE_LIBRARY.values():
        required.update(cols)

    missing = [c for c in required if c not in df.columns]

    if missing:
        raise KeyError(f"Missing required columns in merged: {missing}")


def _to_dataframe(obj):
    if obj is None:
        return pd.DataFrame()

    if isinstance(obj, pd.DataFrame):
        return obj.copy()

    if isinstance(obj, list):
        if len(obj) == 0:
            return pd.DataFrame()

        rows = []
        for x in obj:
            if isinstance(x, dict):
                rows.append(x)
            elif is_dataclass(x):
                rows.append(asdict(x))
            elif hasattr(x, "__dict__"):
                rows.append(vars(x))
            else:
                rows.append(x)

        return pd.DataFrame(rows)

    if isinstance(obj, dict):
        return pd.DataFrame([obj])

    return pd.DataFrame(obj)


def make_sim_config():
    kwargs = dict(
        initial_cash=INITIAL_CASH,
        max_open_trades=1,
        fee_bps=10,
        slippage_bps=1,
        sim_start=SIM_START,
        sim_end=SIM_END,
        sim_tz=tz,
    )

    try:
        return SimConfig(
            **kwargs,
            log_level="WARNING",
            progress=False,
            progress_bar=False,
        )
    except TypeError:
        return SimConfig(**kwargs)


def summarize_sim_result(res, params: dict) -> dict:
    trades = _to_dataframe(getattr(res, "trades", None))

    if trades.empty or "pnl" not in trades.columns:
        closed = pd.DataFrame()
    else:
        closed = trades[trades["pnl"].notna()].copy()

    num_trades = len(closed)

    if num_trades:
        pnl = float(closed["pnl"].sum())
        wins = closed[closed["pnl"] > 0]
        losses = closed[closed["pnl"] <= 0]

        win_rate = float(len(wins) / num_trades * 100)
        avg_pnl = float(closed["pnl"].mean())
        avg_winner = float(wins["pnl"].mean()) if len(wins) else np.nan
        avg_loser = float(losses["pnl"].mean()) if len(losses) else np.nan
        best_trade = float(closed["pnl"].max())
        worst_trade = float(closed["pnl"].min())
    else:
        pnl = 0.0
        win_rate = 0.0
        avg_pnl = 0.0
        avg_winner = np.nan
        avg_loser = np.nan
        best_trade = np.nan
        worst_trade = np.nan

    final_equity = INITIAL_CASH + pnl

    equity_curve = _to_dataframe(getattr(res, "equity_curve", None))
    if not equity_curve.empty:
        for col in ["equity", "balance", "cash"]:
            if col in equity_curve.columns:
                final_equity = float(equity_curve[col].iloc[-1])
                break

    out = {
        **params,
        "num_trades": num_trades,
        "win_rate": win_rate,
        "pnl": pnl,
        "final_equity": final_equity,
        "return_pct": (final_equity / INITIAL_CASH - 1) * 100,
        "avg_pnl": avg_pnl,
        "avg_winner": avg_winner,
        "avg_loser": avg_loser,
        "best_trade": best_trade,
        "worst_trade": worst_trade,
    }

    stats = getattr(res, "stats", {})
    if isinstance(stats, pd.Series):
        stats = stats.to_dict()

    if isinstance(stats, dict):
        for k in [
            "max_drawdown",
            "max_drawdown_pct",
            "profit_factor",
            "total_return",
            "total_return_pct",
        ]:
            if k in stats:
                out[k] = stats[k]

    return out


# ============================================================
# Validate
# ============================================================

validate_columns(merged)


# ============================================================
# Resume support
# ============================================================

if RESULTS_PATH.exists():
    previous = pd.read_csv(RESULTS_PATH)
    done_keys = set(previous["key"].astype(str))
    results = previous.to_dict("records")
    print(f"Resuming | completed={len(done_keys):,}")
else:
    done_keys = set()
    results = []


# ============================================================
# Grid - focused under 5K combinations
# ============================================================

grid = list(itertools.product(
    EMA_FILTER_LIBRARY.keys(),
    MACD_CONFIGS,
    SUPER_OPEN_LIBRARY.keys(),
    CLOSE_RULE_LIBRARY.keys(),
))

expected_combos = (
    len(EMA_FILTER_LIBRARY)
    * len(MACD_CONFIGS)
    * len(SUPER_OPEN_LIBRARY)
    * len(CLOSE_RULE_LIBRARY)
)

print(f"Expected combinations: {expected_combos:,}")
print(f"Total combinations: {len(grid):,}")
print(f"Already completed: {len(done_keys):,}")
print(f"Remaining: {len(grid) - len(done_keys):,}")

# ============================================================
# Simulation strategy uses precomputed open/close signals
# ============================================================

open_rules_long = ALL(
    Rule("Optimized open signal", lambda c: c.flag("__opt_open_signal"))
)

close_rules_long = ALL(
    Rule("Optimized close signal", lambda c: c.flag("__opt_close_signal"))
)

strategy = Strategy(
    open_rules_long=open_rules_long,
    close_rules_long=close_rules_long,
)


# ============================================================
# Run optimizer
# ============================================================

t0 = time.time()
new_runs = 0

for (
    ema_filter_key,
    macd_cfg,
    supertrend_key,
    close_key,
) in tqdm(grid, desc="Optimizing combo strategy"):

    macd_rule_key = macd_cfg["macd_rule_key"]
    macd_cfg_id = macd_cfg["macd_cfg_id"]
    thresholds = macd_cfg["thresholds"]

    key = (
        f"ema_{ema_filter_key}__"
        f"macd_{macd_rule_key}_{macd_cfg_id}__"
        f"st_{supertrend_key}__"
        f"close_{close_key}__"
        f"n{N_CONFIRM}"
    )

    if key in done_keys:
        continue

    ema_signal = close_above_all_cols(
        merged,
        EMA_FILTER_LIBRARY[ema_filter_key],
    )

    green_signal = rolling_all_bool(
        merged["close"].to_numpy(dtype=float) > merged["open"].to_numpy(dtype=float),
        N_CONFIRM,
    )

    macd_signal = build_macd_signal(
        merged,
        macd_rule_key=macd_rule_key,
        thresholds=thresholds,
    )

    supertrend_signal = all_cols_gt_zero(
        merged,
        SUPER_OPEN_LIBRARY[supertrend_key],
    )

    close_signal = any_cols_lt_zero(
        merged,
        CLOSE_RULE_LIBRARY[close_key],
    )

    merged["__opt_open_signal"] = (
        ema_signal
        & green_signal
        & macd_signal
        & supertrend_signal
    )

    merged["__opt_close_signal"] = close_signal

    res = run_simulation(
        df=merged,
        strategy=strategy,
        cfg=make_sim_config(),
        time_col="t",
        price_col="close",
    )

    params = {
        "key": key,
        "ema_filter_key": ema_filter_key,
        "macd_rule_key": macd_rule_key,
        "macd_cfg_id": macd_cfg_id,
        "supertrend_key": supertrend_key,
        "close_key": close_key,
        "n_confirm": N_CONFIRM,

        "macd_1m_threshold": thresholds.get("macd_1m_threshold", np.nan),
        "macd_5m_threshold": thresholds.get("macd_5m_threshold", np.nan),
        "hist_1m_threshold": thresholds.get("hist_1m_threshold", np.nan),
        "hist_5m_threshold": thresholds.get("hist_5m_threshold", np.nan),
        "hist_15m_threshold": thresholds.get("hist_15m_threshold", np.nan),
    }

    results.append(summarize_sim_result(res, params))
    done_keys.add(key)
    new_runs += 1

    if new_runs % CHECKPOINT_EVERY == 0:
        pd.DataFrame(results).to_csv(RESULTS_PATH, index=False)
        elapsed_min = (time.time() - t0) / 60
        print(
            f"Checkpoint saved | new_runs={new_runs:,} | "
            f"total_done={len(done_keys):,} | elapsed={elapsed_min:.1f} min"
        )


# Final save
pd.DataFrame(results).to_csv(RESULTS_PATH, index=False)

print(f"Done. Saved: {RESULTS_PATH}")


# ============================================================
# View best results
# ============================================================

opt = pd.read_csv(RESULTS_PATH)

valid = opt[opt["num_trades"] >= MIN_TRADES_FOR_RANKING].copy()

best_by_profit = valid.sort_values(
    ["pnl", "final_equity", "profit_factor", "win_rate"],
    ascending=False,
)

best_by_winrate = valid.sort_values(
    ["win_rate", "pnl", "num_trades"],
    ascending=False,
)

print("\nBEST BY PROFIT")
display(best_by_profit.head(30))

print("\nBEST BY WIN RATE")
display(best_by_winrate.head(30))

In [ ]:
strategy = Strategy(
    open_rules_long=supertrend_long_filter_strict,
    close_rules_long=close_rules_long,
)

res = run_simulation(
    df=merged,
    strategy=strategy,
    cfg=SimConfig(
        initial_cash=10_000,
        max_open_trades=1,
        fee_bps=10,
        slippage_bps=1,
        # sim_start=sim_start,
        sim_start="2026-01-01",
        sim_end="2026-05-18",
        #sim_start="2026-05-10",
        #sim_end="2026-05-15",
        sim_tz=tz,
    ),
    time_col="t",
    price_col="close",
)

res.stats #, res.events.head(), res.equity_curve.tail()

In [ ]:
trade_summary_table(res.trades, initial_cash=10_000, interval="1m")

In [ ]:
# plot_indicators = make_plot_indicators(
#     base_specs=ind_1m,
#     mtf_specs={
#         "5m": ind_5m,
#         "15m": ind_15m,
#     },
#     mtf_include={
#         "5m": ["macd_8_21_5"],
#         "15m": ["macd_8_21_5"],
#     }
# )

In [ ]:
indicators_by_tf = {
    "1m": ind_1m,
    "5m": ind_5m,
    "15m": ind_15m,
}

plot_toggles = [
    PlotToggle("1m", "ema", visible="legendonly"),
    PlotToggle("5m", "ema", visible="legendonly"),
    PlotToggle("15m", "ema", visible="legendonly"),


    # Available in legend, hidden by default
    PlotToggle("1m", "macd_8_21_5", visible=False),
    PlotToggle("5m", "macd_8_21_5", visible=False),
    PlotToggle("15m", "macd_8_21_5", visible=False),

    PlotToggle("1m", "supertrend_10_3", visible="legendonly"),
    PlotToggle("5m", "supertrend_10_3", visible="legendonly"),
    PlotToggle("15m", "supertrend_10_3", visible="legendonly"),

    # Not plotted at all, but still can be used in rules
    PlotToggle("1m", "rsi14", visible=False),
    PlotToggle("5m", "rsi14", visible=False),
    PlotToggle("15m", "rsi14", visible=False),
]

plot_indicators = make_plot_indicators_from_toggles(
    indicators_by_tf=indicators_by_tf,
    toggles=plot_toggles,
)


assert_plot_columns_exist(merged, plot_indicators)

In [ ]:
plot_simulation(
    df_raw=merged,
    symbol=symbol,
    interval="1m",
    market=market,
    trades=res.trades,
    ma_windows=[],
    indicators=plot_indicators,
    plot_cfg=PlotConfig(
        tz=tz,
        # date_from="2026-05-06",
        # date_to="2026-05-10",
        date_from="2026-1-7",
        date_to="2026-1-12",
        height=1400,
        width=1900,
    ),
)

# plot_simulation(
#     df_raw=df1_feat,  # (ideally raw candles df; but if yours works as-is, ok)
#     symbol=symbol,
#     interval="5m",
#     market=market,
#     trades=res.trades,
#     ma_windows=[20],
#     indicators=ind_5m,
#     plot_cfg=PlotConfig(
#         tz="Asia/Karachi",
#         date_from="2026-02-01",
#         date_to="2026-02-05",
#         height=1400,
#         width=1900
#     ),
# )